In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import re
from pycontrails.core import GeoVectorDataset
from pycontrails.models.gpat.gpat import create_jobs_df, filter_jobs_df, load_fl_df, load_pl_df, load_chem_ds, load_chem_da, plot_heatmap, anim_chem, mc_test, boxm_test

In [2]:
outputs_dir = f"{os.getcwd()}/outputs/"

In [3]:
# Filter criteria
criteria = {
        # "n_ac": 3,
        # "rt_fl": (pd.Timedelta(minutes=30), pd.Timedelta(hours=2)),
        # "date_created": (pd.Timestamp("2024-11-16"), pd.Timestamp("2024-11-17")),
        # "job_id": ['sensitivity_NA_1_1000_0_2_0.01_0.05', 
        #            'sensitivity_NA_2_1000_0_2_0.01_0.05',
        #            'sensitivity_NA_5_1000_0_2_0.01_0.05',]
        "job_id": ['global_jan']

    }

In [4]:
jobs_df = create_jobs_df(outputs_dir)

jobs_df

,t0_sim,rt_sim,ts_sim,lat_bounds,lon_bounds,alt_bounds,hres_sim,vres_sim,eastward_wind,northward_wind,...,n_slices,traj_gen,gen_met,bg_chem,ac_perf,emissions,sim_plumes,plume_to_grid,run_cc,run_boxm
job_id,,,,,,,,,,,,,,,,,,,,,
global_jan,2022-01-01 12:00:00,0 days 00:00:40,0 days 00:00:20,"(-87.5, 87.5)","(-177.5, 177.5)","(8000, 14000)",5.00,1000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mc_v_2_1000_0_10_2_0,2022-01-20 12:00:00,0 days 04:00:00,0 days 00:00:20,"(47.0, 48.0)","(-33.0, -32.0)","(12000, 13000)",0.05,500,0.0,0.0,...,10.0,0.009301,0.272003,0.240219,0.03326,0.047843,0.225385,40.950298,0.000001,0.543526


In [5]:
filtered_df = filter_jobs_df(jobs_df, criteria)

job_ids = filtered_df.index.values
 
filtered_df



,t0_sim,rt_sim,ts_sim,lat_bounds,lon_bounds,alt_bounds,hres_sim,vres_sim,eastward_wind,northward_wind,...,n_slices,traj_gen,gen_met,bg_chem,ac_perf,emissions,sim_plumes,plume_to_grid,run_cc,run_boxm
job_id,,,,,,,,,,,,,,,,,,,,,
global_jan,2022-01-01 12:00:00,0 days 00:00:40,0 days 00:00:20,"(-87.5, 87.5)","(-177.5, 177.5)","(8000, 14000)",5.0,1000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
fl_df = load_fl_df(job_ids, outputs_dir)

fl_df

plume_time_data = GeoVectorDataset(data=fl_df.loc[fl_df["time"] == fl_df["time"].min()])



FileNotFoundError: [Errno 2] No such file or directory: '/home/ktait98/pycontrails_kt/pycontrails/models/gpat/outputs/global_jan/fl_global_jan.pkl'

In [ ]:
pl_df = load_pl_df(job_ids, outputs_dir)
pl_df.columns

pl_df

In [ ]:
chem_ds = load_chem_ds(job_ids, outputs_dir)


In [ ]:
chem_da = []
emi_da = []
# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(10, 6))

for job, ds in enumerate(chem_ds):
    chem_ds[job] = chem_ds[job].assign_coords(species_out=chem_ds[job].attrs["species_out"])
    chem_da.append(chem_ds[job]["Y"])

    emi_da.append(chem_ds[job]["emi"])

    chem_da_cell = chem_da[job].sel(level=chem_da[0].level[1], longitude=-32.9, latitude=47.1, method='nearest')
    emi_da_cell = emi_da[job].sel(level=emi_da[0].level[1], longitude=-32.9, latitude=47.1, method='nearest')

    NO_data = chem_da_cell.sel(species_out="NO")
    NO2_data = chem_da_cell.sel(species_out="NO2")
    O3_data = chem_da_cell.sel(species_out="O3")

    NO_emi_data = emi_da_cell.sel(emi_species="NO")
    NO2_emi_data = emi_da_cell.sel(emi_species="NO2")

    # Plot the time series data
    NO_data.plot(ax=ax, label=f"NO {chem_da_cell['job_id'].values[0]}")
    NO_emi_data.plot(ax=ax, label=f"NO Emissions {emi_da_cell['job_id'].values[0]}")

# Add labels and legend
ax.set_xlabel('Time')
ax.set_ylabel('Concentration')
ax.set_title('Time Series of Species Concentration at Selected Cell')
ax.legend()

# plt.show()
pd.set_option('display.max_rows', 500)
NO_df = NO_data.to_dataframe()
NO_df

NO_data.max()


In [ ]:

# chem_ds_stacked = chem_ds.stack(
#             {"cell": ["level", "longitude", "latitude"]}
#         )
# chem_ds_stacked = chem_ds_stacked.reset_index("cell")

# chem_ds_stacked


In [ ]:

# max_emi_cell = chem_ds_stacked["emi"].mean(dim="time").argmax()#.item()
# print(max_emi_cell)
# # find cell that has max emissions averaged over time in it
# cell_chem_ds = chem_ds_stacked.sel(job_id=job_ids[0], cell=max_emi_cell)
# cell_chem_ds
# # Select the emissions for the specified species
# emi_data = cell_chem_ds["emi"].sel(emi_species="NO")
# chem_data = cell_chem_ds["Y"].sel(species_out="NO")
# emi_data.plot()
# chem_data.plot()

# # # Convert time and emi data to pandas Series
# # ts = 0
# # time_series = pd.Series(emi_data["time"].values)
# # emi_series = pd.Series(emi_data.values)

# # # Print time and emi values side by side
# # for time, emi in zip(time_series, emi_series):
# #     ts += 1
# #     print(f"TS: {ts}, Time: {time}, EMI: {emi}")

# for s, species in enumerate(chem_ds["species"].values):
#     print(chem_ds["bg_chem"].isel(level=1,latitude=0,longitude=0, species=s).values)



In [ ]:
anim_chem(job_ids[0], jobs_df, fl_df, pl_df, chem_ds[0], var1="emi", var2="NO", level=chem_da[0].level[1], resample_freq="2min")

In [ ]:
# vecmass, gridmass, mc = mc_test(job_ids[0], jobs_df, fl_df, pl_df, chem_ds)
# mc

In [ ]:
path = '/user/work/kt16229/pycontrails_kt/pycontrails/models/gpat/'

cell_chem_ds = boxm_test(path, job_ids[0], 0, chem_ds)

# dj_data = cell_chem_ds["DJ"].sel(photol_coeffs=3)
# dj_orig_data = cell_chem_ds["DJ_orig"].sel(photol_coeffs=3)
# dj_data.plot()
# dj_orig_data.plot()

chem_data = cell_chem_ds["Y"].sel(species_out="NO")
chem_orig_data = cell_chem_ds["Y_orig"].sel(species_out="NO")
chem_data.plot()
chem_orig_data.plot()
